# 10. Accessors Overview — 18개 accessor 한눈에

hwpapi 에는 **18개 accessor** 가 도메인별로 그룹핑되어 있습니다.

## 🚀 TL;DR — Discovery

```python
app = App()
app.help()               # 18개 accessor + context manager 출력
repr(app)                # App(visible=True, version='13.0.0', docs=2, page=5/20)
```

실제 `app.help()` 출력 모습:


In [ ]:
import sys
sys.path.insert(0, '.')
from tutorial_helpers import run_and_show

In [ ]:
# app.help() 를 문서에 삽입
import io, contextlib
from hwpapi.core import App
buf = io.StringIO()
a = App(is_visible=False)
with contextlib.redirect_stdout(buf):
    a.help()
output = buf.getvalue()
# HWP 앱 종료
try: a.quit()
except Exception: pass
print(output)

## 18개 accessor 매트릭스

| 카테고리 | Accessor | 용도 |
|:---|:---|:---|
| **Navigation & Selection** | `app.move` | 커서 이동 |
| | `app.sel` | 선택 제어 |
| **Collections** | `app.documents` | 열린 문서 |
| | `app.fields` | 누름틀 |
| | `app.bookmarks` | 책갈피 |
| | `app.hyperlinks` | 하이퍼링크 |
| | `app.images` | 이미지 |
| | `app.styles` | 문단 스타일 |
| | `app.controls` | 모든 control |
| **Structure** | `app.cell` | 표 셀 |
| | `app.table` | 표 구조 + 배치 서식 |
| | `app.page` | 페이지 설정 |
| **Transform & View** | `app.convert` | 숫자/폰트/줄 나눔 |
| | `app.view` | zoom, 모드 |
| **Quality & Templates** | `app.lint` | 문서 품질 체크 |
| | `app.template` | 템플릿 save/apply |
| | `app.config` | App 선호도 |
| **Presets & Debug** | `app.preset` | 꾸미기 프리셋 11종 |
| | `app.debug` | 디버깅 도구 |


## 주요 예제

### 1. Fields — dict-style mail merge

```python
app.fields["name"] = "홍길동"
app.fields.update({"date": "2026-04-15", "amount": "1,200,000"})
print(app.fields.to_dict())
# {'name': '홍길동', 'date': '2026-04-15', 'amount': '1,200,000'}
```

실제 렌더링 결과:


In [ ]:
run_and_show("fields_dict_full", '''
app.set_charshape(bold=True, height=1600)
app.insert_text("계약서\n\n")
app.set_charshape(bold=False, height=1000)
app.insert_text("계약자: {{name}}\n")
app.insert_text("계약일: {{date}}\n")
app.insert_text("계약 금액: {{amount}}원\n\n")
app.move.top_of_file()
app.fields.from_brackets()
app.fields.update({
    "name": "홍길동",
    "date": "2026-04-15",
    "amount": "1,200,000",
})
''')

*위: `{{tag}}` → 필드 자동 변환 → dict-style 일괄 값 주입 결과*

### 2. Table batch formatting

```python
app.insert_table(rows=3, cols=4)
app.table.header_row(color="sky", text_color="#FFFFFF")
app.table.current_row(bg="#FFFFCC")
app.table.align(horz="right", scope="current_col")
```

실제 렌더링 결과:


In [ ]:
run_and_show("table_batch_ops", '''
data = [["항목", "Q1", "Q2", "Q3"],
        ["매출", "100", "120", "140"],
        ["비용", "60", "65", "70"],
        ["이익", "40", "55", "70"]]
app.insert_table(data=data)
app.move.top_of_file()
for _ in range(3):
    app.api.Run("TableLowerCell")
app.preset.table_header(color="sky", text_color="#FFFFFF", bold=True)
app.move.bottom_of_file()
''')

*위: `table.header_row()` 스카이블루 제목행 적용 결과*

### 3. Selection + compress/expand

```python
app.sel.current_paragraph()
app.sel.compress(step=2)       # 자간·장평 동시 축소
```


In [ ]:
run_and_show("sel_operations", '''
app.set_charshape(bold=True, height=1600)
app.insert_text("app.sel — 선택 동작\n\n")
app.set_charshape(bold=False, height=1100)
app.insert_text("이 문단은 커서 위치에서 current_paragraph() 로\n")
app.insert_text("문단 전체를 선택한 뒤 서식 적용 가능합니다.\n")
app.insert_text("예: 굵게, 색 변경, 형광펜, 자간 조정.\n\n")
# Select one paragraph and make it bold+red
app.move.top_of_file()
app.api.Run("MoveDown")
app.api.Run("MoveDown")
app.sel.current_paragraph()
app.set_charshape(bold=True, text_color="#FF6600")
app.api.Run("Cancel")
''')

*위: `sel.current_paragraph()` 후 서식 적용 결과 (한 문단만 강조)*

### 4. Context manager 사용

```python
with app.batch_mode():                       # 5~10배 가속
    for row in df.iterrows():
        app.fields.update(row.to_dict())
        app.save(f"out/{row['name']}.hwp")

with app.charshape_scope(bold=True, text_color="#FF0000"):
    app.insert_text("강조 텍스트")
# 블록 종료 시 이전 charshape 로 자동 복원
```

실제 `charshape_scope` 동작:


In [ ]:
run_and_show("charshape_scope_real", '''
app.insert_text("앞 텍스트 — ")
with app.charshape_scope(bold=True, text_color="#FF0000", height=1500):
    app.insert_text("강조 부분")
app.insert_text(" — 뒤 텍스트 (기본 서식 자동 복원)\n")
''')

*위: `charshape_scope` 블록 내부만 굵은 빨간색 큰 글씨, 블록 종료 시 자동 복원*

## 마치며

각 accessor 의 자세한 사용법과 시각 예제:
- 프리셋 → [11_presets_gallery](11_presets_gallery.ipynb)
- 대량 처리 / workflow → [12_batch_and_workflow](12_batch_and_workflow.ipynb)
- 디버깅 / 품질 / 설정 → [13_debugging_tools](13_debugging_tools.ipynb)
- 레거시 → 신 API 마이그레이션 표 → [docs/API_GUIDE.md](../../docs/API_GUIDE.md)
